In [4]:
import cv2
from ultralytics import YOLO

model = YOLO("yolov8s.pt") 

cap = cv2.VideoCapture("foot ball.mp4")


fps = cap.get(cv2.CAP_PROP_FPS)
delay = int(1000 / fps)


frame_skip = 1
frame_count = 0


track_history = {}

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

   
    if frame_count % frame_skip != 0:
        continue

    frame = cv2.resize(frame, (960, 540))

    results = model.track(
        frame,
        persist=True,
        conf=0.4,
        tracker="bytetrack.yaml"
    )


    if results[0].boxes is not None:
        boxes = results[0].boxes

        coords = boxes.xyxy.cpu().numpy()
        classes = boxes.cls.cpu().numpy()
        ids = boxes.id

        if ids is not None:
            ids = ids.cpu().numpy()

        for i, (box, cls) in enumerate(zip(coords, classes)):

           
            if int(cls) != 0:
                continue

            x1, y1, x2, y2 = map(int, box)

            track_id = int(ids[i]) if ids is not None else -1

            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            cv2.putText(frame,
                        f"ID {track_id}",
                        (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6,
                        (0, 255, 0),
                        2)

            cx, cy = (x1 + x2) // 2, (y1 + y2) // 2

            if track_id not in track_history:
                track_history[track_id] = []

            track_history[track_id].append((cx, cy))

            # Limit trail length
            if len(track_history[track_id]) > 30:
                track_history[track_id].pop(0)

            for j in range(1, len(track_history[track_id])):
                cv2.line(frame,
                         track_history[track_id][j - 1],
                         track_history[track_id][j],
                         (0, 255, 0),
                         2)

    cv2.imshow("Multi-Object Tracking", frame)

 
    if cv2.waitKey(delay) == 27:
        break

cap.release()
cv2.destroyAllWindows()


0: 384x640 2 persons, 1 umbrella, 281.4ms
Speed: 4.6ms preprocess, 281.4ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 287.7ms
Speed: 7.4ms preprocess, 287.7ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 230.3ms
Speed: 5.2ms preprocess, 230.3ms inference, 2.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 268.0ms
Speed: 5.5ms preprocess, 268.0ms inference, 5.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 237.9ms
Speed: 5.4ms preprocess, 237.9ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 231.0ms
Speed: 4.4ms preprocess, 231.0ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 210.8ms
Speed: 4.6ms preprocess, 210.8ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 persons, 302.9ms
Speed: 4.9ms preprocess, 302.9ms inference, 1.9ms post